# 01 - 强化学习基础


强化学习研究的是智能体如何通过与环境交互来学习行为策略。和监督学习不同，强化学习没有每一步的标准答案，只有环境给出的奖励信号；智能体需要在试错中学会最大化长期回报。

学习目标：

- 理解 Agent、Environment、State、Action、Reward、Policy 的关系。
- 掌握 MDP 的五元组：$S, A, P, R, \gamma$。
- 理解价值函数、动作价值函数和贝尔曼方程。
- 能看懂价值迭代如何在已知环境模型时求解最优策略。
- 能理解 Q-Learning 如何在未知转移概率时通过交互学习。

## 环境准备与导入

这一段保留原教程的导入、标题打印和基础类/函数依赖。先运行它，后续代码单元会复用这里导入的库与随机种子。

In [1]:
"""
第六章 6.1：强化学习基础
========================

强化学习 (Reinforcement Learning, RL) 是机器学习的第三大范式：
  - 监督学习: 从标注数据中学习
  - 无监督学习: 从无标注数据中发现结构
  - 强化学习: 通过与环境交互，从奖励信号中学习

应用场景: 游戏AI、机器人控制、推荐系统、自动驾驶、RLHF(LLM对齐)

本节内容：
1. RL 的核心概念
2. 马尔可夫决策过程 (MDP)
3. 价值函数与贝尔曼方程
4. 动态规划求解
5. Q-Learning (表格法)
6. 实战: 网格世界
"""

import numpy as np

print("=" * 60)
print("第六章 6.1：强化学习基础")
print("=" * 60)

第六章 6.1：强化学习基础


## 1. 核心概念


强化学习的核心不是“预测正确答案”，而是“通过行动影响未来”。智能体每做一个动作，环境会返回新状态和奖励；当前动作可能短期看起来不好，但长期可能更好，这就是 RL 比普通分类/回归更难的地方。

初学时建议先抓住一个闭环：状态告诉智能体现在在哪里，策略决定做什么动作，环境返回奖励和下一个状态，智能体再根据结果更新策略。

In [2]:
print("\n" + "=" * 60)
print("1. 强化学习的核心概念")
print("=" * 60)

print("""
【强化学习 vs 监督学习】

监督学习:
  数据集 → {(输入, 正确答案)} → 模型学习映射
  "老师给你正确答案"

强化学习:
  智能体(Agent) 在环境(Environment) 中行动
  环境给出奖励(Reward)信号
  智能体学习最大化长期累积奖励的策略
  "没人告诉你正确答案，你通过试错来学习"

【核心要素】

  ┌─────────────────────────────────────┐
  │           环境 (Environment)         │
  │                                      │
  │  状态 s ──→ 智能体(Agent) ──→ 动作 a │
  │       ↑                         │    │
  │       └── 奖励 r, 新状态 s' ←──┘    │
  └─────────────────────────────────────┘

  - 状态 (State, s): 环境当前的情况
  - 动作 (Action, a): 智能体可以做的选择
  - 奖励 (Reward, r): 环境对动作的即时反馈
  - 策略 (Policy, π): 状态 → 动作的映射规则
  - 目标: 找到最优策略 π*，最大化累积奖励

【例子：下棋】
  状态 = 棋盘局面
  动作 = 落子位置
  奖励 = 赢了+1, 输了-1, 其他0
  策略 = 看到某个局面，决定下哪里

【挑战】
  1. 延迟奖励: 当前动作的好坏可能要很久以后才知道
  2. 探索 vs 利用: 是尝试新动作，还是用已知的好动作？
  3. 信用分配: 一局棋赢了，到底是哪步棋下得好？
""")


1. 强化学习的核心概念

【强化学习 vs 监督学习】

监督学习:
  数据集 → {(输入, 正确答案)} → 模型学习映射
  "老师给你正确答案"

强化学习:
  智能体(Agent) 在环境(Environment) 中行动
  环境给出奖励(Reward)信号
  智能体学习最大化长期累积奖励的策略
  "没人告诉你正确答案，你通过试错来学习"

【核心要素】

  ┌─────────────────────────────────────┐
  │           环境 (Environment)         │
  │                                      │
  │  状态 s ──→ 智能体(Agent) ──→ 动作 a │
  │       ↑                         │    │
  │       └── 奖励 r, 新状态 s' ←──┘    │
  └─────────────────────────────────────┘

  - 状态 (State, s): 环境当前的情况
  - 动作 (Action, a): 智能体可以做的选择
  - 奖励 (Reward, r): 环境对动作的即时反馈
  - 策略 (Policy, π): 状态 → 动作的映射规则
  - 目标: 找到最优策略 π*，最大化累积奖励

【例子：下棋】
  状态 = 棋盘局面
  动作 = 落子位置
  奖励 = 赢了+1, 输了-1, 其他0
  策略 = 看到某个局面，决定下哪里

【挑战】
  1. 延迟奖励: 当前动作的好坏可能要很久以后才知道
  2. 探索 vs 利用: 是尝试新动作，还是用已知的好动作？
  3. 信用分配: 一局棋赢了，到底是哪步棋下得好？



## 2. 马尔可夫决策过程 (MDP)


MDP 是强化学习的数学语言。马尔可夫性质的含义是：如果当前状态描述足够完整，那么未来只依赖当前状态和动作，不需要回看全部历史。

折扣因子 $\gamma$ 控制模型有多重视未来。$\gamma$ 越接近 1，智能体越愿意为了长期收益忍受短期损失；$\gamma$ 越小，智能体越短视。

In [3]:
print("\n" + "=" * 60)
print("2. 马尔可夫决策过程 (MDP)")
print("=" * 60)

print("""
【MDP 是 RL 的数学框架】

定义: MDP = (S, A, P, R, γ)
  S: 状态空间 (所有可能的状态)
  A: 动作空间 (所有可能的动作)
  P: 状态转移概率 P(s'|s, a) (做了动作后到哪个新状态)
  R: 奖励函数 R(s, a, s') (获得多少奖励)
  γ: 折扣因子 (0≤γ≤1, 未来奖励的衰减)

【马尔可夫性质】
  下一个状态只取决于当前状态和动作，与历史无关。
  P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, ...) = P(s_{t+1} | s_t, a_t)

【折扣因子 γ 的意义】
  累积奖励 G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...
  
  γ = 0: 只看当前奖励（短视）
  γ = 1: 未来奖励和当前同等重要
  γ = 0.99: 常用值，稍微偏好近期奖励

  为什么要折扣？
  1. 数学上保证累积奖励有限
  2. 现实中，未来有不确定性，近期奖励更可靠
""")

# 定义一个简单的网格世界
print("\n--- 简单网格世界 ---")
print("""
4×4 网格世界:
  ┌───┬───┬───┬───┐
  │ S │   │   │ G │   S=起点, G=终点(奖励+1)
  ├───┼───┼───┼───┤
  │   │ X │   │   │   X=陷阱(奖励-1)
  ├───┼───┼───┼───┤
  │   │   │   │ X │
  ├───┼───┼───┼───┤
  │   │   │   │   │
  └───┴───┴───┴───┘

动作: 上(0), 下(1), 左(2), 右(3)
到达 G 或 X 则结束。每步奖励 -0.04（鼓励快速到达目标）
""")


class GridWorld:
    """简单的网格世界环境"""
    
    def __init__(self, size=4):
        self.size = size
        self.start = (0, 0)
        self.goal = (0, 3)       # 目标: +1
        self.traps = [(1, 1), (2, 3)]  # 陷阱: -1
        self.state = self.start
        
        # 动作: 上下左右
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.action_names = ['上', '下', '左', '右']
        self.n_actions = 4
        self.n_states = size * size
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        """执行动作，返回 (新状态, 奖励, 是否结束)"""
        dy, dx = self.actions[action]
        new_y = max(0, min(self.size - 1, self.state[0] + dy))
        new_x = max(0, min(self.size - 1, self.state[1] + dx))
        self.state = (new_y, new_x)
        
        if self.state == self.goal:
            return self.state, 1.0, True
        elif self.state in self.traps:
            return self.state, -1.0, True
        else:
            return self.state, -0.04, False
    
    def state_to_idx(self, state):
        return state[0] * self.size + state[1]
    
    def idx_to_state(self, idx):
        return (idx // self.size, idx % self.size)


env = GridWorld()
print(f"状态空间大小: {env.n_states}")
print(f"动作空间大小: {env.n_actions}")


2. 马尔可夫决策过程 (MDP)

【MDP 是 RL 的数学框架】

定义: MDP = (S, A, P, R, γ)
  S: 状态空间 (所有可能的状态)
  A: 动作空间 (所有可能的动作)
  P: 状态转移概率 P(s'|s, a) (做了动作后到哪个新状态)
  R: 奖励函数 R(s, a, s') (获得多少奖励)
  γ: 折扣因子 (0≤γ≤1, 未来奖励的衰减)

【马尔可夫性质】
  下一个状态只取决于当前状态和动作，与历史无关。
  P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, ...) = P(s_{t+1} | s_t, a_t)

【折扣因子 γ 的意义】
  累积奖励 G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...

  γ = 0: 只看当前奖励（短视）
  γ = 1: 未来奖励和当前同等重要
  γ = 0.99: 常用值，稍微偏好近期奖励

  为什么要折扣？
  1. 数学上保证累积奖励有限
  2. 现实中，未来有不确定性，近期奖励更可靠


--- 简单网格世界 ---

4×4 网格世界:
  ┌───┬───┬───┬───┐
  │ S │   │   │ G │   S=起点, G=终点(奖励+1)
  ├───┼───┼───┼───┤
  │   │ X │   │   │   X=陷阱(奖励-1)
  ├───┼───┼───┼───┤
  │   │   │   │ X │
  ├───┼───┼───┼───┤
  │   │   │   │   │
  └───┴───┴───┴───┘

动作: 上(0), 下(1), 左(2), 右(3)
到达 G 或 X 则结束。每步奖励 -0.04（鼓励快速到达目标）

状态空间大小: 16
动作空间大小: 4


## 3. 价值函数


价值函数回答“这个状态值得期待吗”。动作价值函数 Q 则进一步回答“在这个状态做某个动作值得期待吗”。

贝尔曼方程把长期价值拆成两部分：即时奖励 + 下一个状态的价值。这种递归关系是动态规划、Q-Learning 和许多深度 RL 算法的共同基础。

In [4]:
print("\n" + "=" * 60)
print("3. 价值函数与贝尔曼方程")
print("=" * 60)

print("""
【价值函数: 评估"一个状态有多好"】

状态价值函数 V^π(s):
  从状态 s 出发，按照策略 π 行动，期望的累积奖励
  V^π(s) = E[r_t + γ·r_{t+1} + γ²·r_{t+2} + ... | s_t = s, π]

动作价值函数 Q^π(s, a):
  在状态 s 执行动作 a，然后按策略 π 行动的期望累积奖励
  Q^π(s, a) = E[r_t + γ·r_{t+1} + ... | s_t = s, a_t = a, π]

【贝尔曼方程 (递归关系)】

  V^π(s) = Σ_a π(a|s) · Σ_{s'} P(s'|s,a) · [R(s,a,s') + γ·V^π(s')]
  
  直觉: 当前状态的价值 = 即时奖励 + 折扣后的下一个状态的价值

  这是一个递推公式，可以通过迭代求解！

【最优价值函数】
  V*(s) = max_a Σ_{s'} P(s'|s,a) · [R(s,a,s') + γ·V*(s')]
  
  在每个状态都选最优动作时的价值。
""")


3. 价值函数与贝尔曼方程

【价值函数: 评估"一个状态有多好"】

状态价值函数 V^π(s):
  从状态 s 出发，按照策略 π 行动，期望的累积奖励
  V^π(s) = E[r_t + γ·r_{t+1} + γ²·r_{t+2} + ... | s_t = s, π]

动作价值函数 Q^π(s, a):
  在状态 s 执行动作 a，然后按策略 π 行动的期望累积奖励
  Q^π(s, a) = E[r_t + γ·r_{t+1} + ... | s_t = s, a_t = a, π]

【贝尔曼方程 (递归关系)】

  V^π(s) = Σ_a π(a|s) · Σ_{s'} P(s'|s,a) · [R(s,a,s') + γ·V^π(s')]

  直觉: 当前状态的价值 = 即时奖励 + 折扣后的下一个状态的价值

  这是一个递推公式，可以通过迭代求解！

【最优价值函数】
  V*(s) = max_a Σ_{s'} P(s'|s,a) · [R(s,a,s') + γ·V*(s')]

  在每个状态都选最优动作时的价值。



## 4. 价值迭代 (Value Iteration)


价值迭代适用于环境规则已知的情况。它不断用贝尔曼最优方程更新每个状态的价值，直到价值不再明显变化。

在网格世界中，你可以把它想成从终点和陷阱向外传播价值：靠近目标的状态价值变高，靠近陷阱的状态价值变低，最终每个格子都能选出最优方向。

In [5]:
print("\n" + "=" * 60)
print("4. 价值迭代 (Value Iteration)")
print("=" * 60)

print("""
【算法】
  1. 初始化 V(s) = 0 对所有 s
  2. 重复直到收敛:
     对每个状态 s:
       V(s) = max_a [R(s,a) + γ · V(s')]
  3. 从最优价值提取策略:
     π*(s) = argmax_a [R(s,a) + γ · V(s')]
""")

def value_iteration(env, gamma=0.9, threshold=1e-6):
    """价值迭代算法"""
    V = np.zeros(env.n_states)
    
    for iteration in range(1000):
        V_new = np.zeros(env.n_states)
        
        for s_idx in range(env.n_states):
            state = env.idx_to_state(s_idx)
            
            # 终止状态价值为 0
            if state == env.goal or state in env.traps:
                continue
            
            # 对每个动作计算价值
            action_values = []
            for a in range(env.n_actions):
                env.state = state
                next_state, reward, done = env.step(a)
                next_idx = env.state_to_idx(next_state)
                
                if done:
                    action_values.append(reward)
                else:
                    action_values.append(reward + gamma * V[next_idx])
            
            V_new[s_idx] = max(action_values)
        
        # 检查收敛
        if np.max(np.abs(V_new - V)) < threshold:
            print(f"  价值迭代在第 {iteration+1} 次收敛")
            break
        V = V_new
    
    # 提取策略
    policy = np.zeros(env.n_states, dtype=int)
    for s_idx in range(env.n_states):
        state = env.idx_to_state(s_idx)
        if state == env.goal or state in env.traps:
            continue
        
        action_values = []
        for a in range(env.n_actions):
            env.state = state
            next_state, reward, done = env.step(a)
            next_idx = env.state_to_idx(next_state)
            if done:
                action_values.append(reward)
            else:
                action_values.append(reward + gamma * V[next_idx])
        
        policy[s_idx] = np.argmax(action_values)
    
    return V, policy

V_opt, policy_opt = value_iteration(env)

# 显示结果
print(f"\n最优状态价值 V*(s):")
V_grid = V_opt.reshape(4, 4)
for i in range(4):
    row = ""
    for j in range(4):
        row += f"{V_grid[i,j]:6.2f} "
    print(f"  {row}")

print(f"\n最优策略 (↑↓←→):")
arrows = ['↑', '↓', '←', '→']
for i in range(4):
    row = ""
    for j in range(4):
        state = (i, j)
        if state == env.goal:
            row += "  G   "
        elif state in env.traps:
            row += "  X   "
        else:
            idx = env.state_to_idx(state)
            row += f"  {arrows[policy_opt[idx]]}   "
    print(f"  {row}")


4. 价值迭代 (Value Iteration)

【算法】
  1. 初始化 V(s) = 0 对所有 s
  2. 重复直到收敛:
     对每个状态 s:
       V(s) = max_a [R(s,a) + γ · V(s')]
  3. 从最优价值提取策略:
     π*(s) = argmax_a [R(s,a) + γ · V(s')]

  价值迭代在第 7 次收敛

最优状态价值 V*(s):
    0.73   0.86   1.00   0.00 
    0.62   0.00   0.86   1.00 
    0.52   0.62   0.73   0.00 
    0.43   0.52   0.62   0.52 

最优策略 (↑↓←→):
    →     →     →     G   
    ↑     X     ↑     ↑   
    ↑     →     ↑     X   
    ↑     ↑     ↑     ←   


## 5. Q-Learning


Q-Learning 不需要知道完整环境模型。它通过真实交互收集 `(s, a, r, s')`，再用 TD target 更新 Q 表。

TD error 表示当前估计和新观察到的目标之间差多少。学习过程就是不断缩小这个差距。$\epsilon$-贪心让智能体既能探索未知动作，也能利用当前最优动作。

In [6]:
print("\n" + "=" * 60)
print("5. Q-Learning (无模型强化学习)")
print("=" * 60)

print("""
【为什么需要 Q-Learning？】
价值迭代需要知道环境的转移概率 P(s'|s,a) → 需要"模型"
但很多时候我们不知道环境的具体规则！

Q-Learning: 通过与环境交互来学习，不需要知道环境模型。
这是"无模型" (Model-Free) 方法。

【Q-Learning 算法】
  初始化 Q(s, a) = 0
  重复:
    1. 在状态 s，选择动作 a (ε-贪心)
    2. 执行 a，观察奖励 r 和新状态 s'
    3. 更新:
       Q(s, a) ← Q(s, a) + α · [r + γ · max_a' Q(s', a') - Q(s, a)]
    4. s ← s'

  α: 学习率
  γ: 折扣因子
  ε: 探索率 (以 ε 的概率随机探索，1-ε 的概率利用已知最优)

【ε-贪心策略 (Exploration vs Exploitation)】
  以概率 ε 随机选动作（探索新的可能）
  以概率 1-ε 选 Q 值最大的动作（利用已知信息）
  
  通常 ε 随训练逐渐减小（开始多探索，后来多利用）
""")

def q_learning(env, n_episodes=1000, alpha=0.1, gamma=0.9, 
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """Q-Learning 算法"""
    Q = np.zeros((env.n_states, env.n_actions))
    epsilon = epsilon_start
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        for step in range(100):  # 最多100步
            s_idx = env.state_to_idx(state)
            
            # ε-贪心选择动作
            if np.random.random() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q[s_idx])
            
            # 执行动作
            next_state, reward, done = env.step(action)
            next_idx = env.state_to_idx(next_state)
            total_reward += reward
            
            # Q-Learning 更新
            best_next_q = np.max(Q[next_idx]) if not done else 0
            td_target = reward + gamma * best_next_q
            td_error = td_target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td_error
            
            if done:
                break
            state = next_state
        
        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, rewards_history

np.random.seed(42)
Q, rewards = q_learning(env, n_episodes=2000)

# 显示学到的 Q 值和策略
print(f"\n--- Q-Learning 结果 ---")
print(f"训练 2000 个 episode")
print(f"最后100个episode平均奖励: {np.mean(rewards[-100:]):.3f}")

# 提取策略
q_policy = np.argmax(Q, axis=1)
print(f"\n学到的策略:")
arrows = ['↑', '↓', '←', '→']
for i in range(4):
    row = ""
    for j in range(4):
        state = (i, j)
        if state == env.goal:
            row += "  G   "
        elif state in env.traps:
            row += "  X   "
        else:
            idx = env.state_to_idx(state)
            row += f"  {arrows[q_policy[idx]]}   "
    print(f"  {row}")

# 验证策略：从起点走到终点
print(f"\n--- 用学到的策略走一遍 ---")
state = env.reset()
path = [state]
for step in range(20):
    s_idx = env.state_to_idx(state)
    action = np.argmax(Q[s_idx])
    state, reward, done = env.step(action)
    path.append(state)
    if done:
        break

print(f"路径: {' → '.join([str(s) for s in path])}")
if state == env.goal:
    print(f"✓ 成功到达目标！步数: {len(path)-1}")
else:
    print(f"到达: {state}")

# 验证Q值合理性
assert Q[env.state_to_idx((0, 2)), 3] > 0, "靠近目标的Q值应该为正"
print("✓ Q-Learning 学到了合理的策略")


5. Q-Learning (无模型强化学习)

【为什么需要 Q-Learning？】
价值迭代需要知道环境的转移概率 P(s'|s,a) → 需要"模型"
但很多时候我们不知道环境的具体规则！

Q-Learning: 通过与环境交互来学习，不需要知道环境模型。
这是"无模型" (Model-Free) 方法。

【Q-Learning 算法】
  初始化 Q(s, a) = 0
  重复:
    1. 在状态 s，选择动作 a (ε-贪心)
    2. 执行 a，观察奖励 r 和新状态 s'
    3. 更新:
       Q(s, a) ← Q(s, a) + α · [r + γ · max_a' Q(s', a') - Q(s, a)]
    4. s ← s'

  α: 学习率
  γ: 折扣因子
  ε: 探索率 (以 ε 的概率随机探索，1-ε 的概率利用已知最优)

【ε-贪心策略 (Exploration vs Exploitation)】
  以概率 ε 随机选动作（探索新的可能）
  以概率 1-ε 选 Q 值最大的动作（利用已知信息）

  通常 ε 随训练逐渐减小（开始多探索，后来多利用）


--- Q-Learning 结果 ---
训练 2000 个 episode
最后100个episode平均奖励: 0.900

学到的策略:
    →     →     →     G   
    ↑     X     ↑     ↑   
    ↑     ←     ↑     X   
    ↑     ↑     ↓     ↓   

--- 用学到的策略走一遍 ---
路径: (0, 0) → (0, 1) → (0, 2) → (0, 3)
✓ 成功到达目标！步数: 3
✓ Q-Learning 学到了合理的策略


## 6. 总结


这一节把强化学习的基本骨架串起来：MDP 定义问题，价值函数评估长期收益，贝尔曼方程建立递推关系，价值迭代用于已知模型，Q-Learning 用于无模型交互学习。

后续深度强化学习会把 Q 表或策略表换成神经网络，但“状态、动作、奖励、价值、探索”这些核心概念不会变。

In [7]:
print("\n" + "=" * 60)
print("本节总结")
print("=" * 60)
print("""
关键要点：
1. RL = 智能体通过与环境交互、从奖励中学习最优行为
2. MDP = (状态, 动作, 转移概率, 奖励, 折扣因子) 
3. 价值函数 V(s)/Q(s,a) 评估状态/动作的长期价值
4. 贝尔曼方程: 当前价值 = 即时奖励 + 折扣×未来价值
5. Q-Learning: 无模型、通过试错学习 Q 值
6. ε-贪心: 平衡探索与利用

RL 方法分类:
  - 基于价值: Q-Learning, DQN
  - 基于策略: REINFORCE, PPO
  - Actor-Critic: A2C, A3C, SAC

下一节: 深度强化学习 → DQN、策略梯度、PPO
""")


本节总结

关键要点：
1. RL = 智能体通过与环境交互、从奖励中学习最优行为
2. MDP = (状态, 动作, 转移概率, 奖励, 折扣因子) 
3. 价值函数 V(s)/Q(s,a) 评估状态/动作的长期价值
4. 贝尔曼方程: 当前价值 = 即时奖励 + 折扣×未来价值
5. Q-Learning: 无模型、通过试错学习 Q 值
6. ε-贪心: 平衡探索与利用

RL 方法分类:
  - 基于价值: Q-Learning, DQN
  - 基于策略: REINFORCE, PPO
  - Actor-Critic: A2C, A3C, SAC

下一节: 深度强化学习 → DQN、策略梯度、PPO



## 深入理解：探索与利用为什么难

如果智能体只利用当前看起来最好的动作，它可能永远发现不了更优路径。如果智能体一直随机探索，它又无法稳定获得高奖励。探索与利用的平衡，是强化学习最经典的难题之一。

在 Q-Learning 中，$\epsilon$-贪心是最简单的解决方式：训练早期 $\epsilon$ 大，多探索；训练后期 $\epsilon$ 小，多利用。真实问题中还会使用 UCB、熵正则、噪声策略等更复杂方法。

判断探索是否足够，可以观察不同动作是否都被尝试过、episode 回报是否长期停滞、策略是否过早陷入某条固定路径。

## 常见误区

1. **奖励不等于价值**：奖励是当前一步的反馈，价值是未来累计回报的期望。
2. **贪心策略不一定一开始就好**：早期 Q 值还不准，必须探索。
3. **折扣因子不是学习率**：$\gamma$ 控制未来奖励权重，$\alpha$ 控制每次更新幅度。
4. **终止状态价值通常设为 0**：因为 episode 已结束，没有未来奖励可加。
5. **Q-Learning 是 off-policy**：它学习的是最优贪心策略，即使行为策略包含探索。

## 课后练习

建议你尝试修改网格世界：移动目标位置、增加陷阱、改变每步惩罚，观察最优策略如何变化。然后修改 $\gamma$、$\alpha$、$\epsilon$ 衰减速度，比较 Q-Learning 收敛速度和最终路径。

这些实验能帮助你形成 RL 直觉：奖励设计会强烈影响行为；探索不足会导致局部最优；折扣因子会改变智能体对远期目标的耐心。

## 核心术语对照

- **Agent**：智能体，负责观察状态并选择动作。
- **Environment**：环境，接收动作并返回奖励和新状态。
- **Policy**：策略，从状态到动作或动作概率的规则。
- **Return**：累计折扣奖励。
- **Value Function**：状态的长期价值估计。
- **Q Function**：状态-动作对的长期价值估计。
- **TD Error**：当前估计与贝尔曼目标之间的差。

## 深入理解：奖励设计决定学习方向

强化学习里，智能体不会自动理解“人类真正想要什么”，它只会优化你给出的奖励。因此奖励设计非常关键。

在网格世界中，目标格给 +1，陷阱给 -1，每走一步给 -0.04。这个每步惩罚看起来很小，却会改变行为：它鼓励智能体尽快到达目标，而不是在安全区域里绕圈。

如果把每步惩罚改成 0，智能体可能不在乎路径长短；如果惩罚太大，智能体可能过度害怕探索。真实任务中也一样：奖励太稀疏会学得慢，奖励太密集又可能诱导出不符合真实目标的捷径。

## 深入理解：TD 更新在做什么

Q-Learning 的更新式可以拆成三部分：

$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'}Q(s',a') - Q(s,a)]$$

其中 $r + \gamma \max_{a'}Q(s',a')$ 是新的目标估计，$Q(s,a)$ 是旧估计，两者差值叫 TD error。

如果 TD error 为正，说明实际观察到的结果比原来预期更好，就提高这个动作的 Q 值。如果 TD error 为负，说明结果比预期差，就降低 Q 值。强化学习就是通过大量这样的局部修正，逐渐形成全局策略。

## 如何判断 RL 算法是否学到了东西

不要只看最后一条路径。更可靠的信号包括：

1. episode 平均奖励是否随训练上升。
2. 探索率降低后，策略是否稳定。
3. 起点附近的 Q 值是否把智能体引向目标。
4. 靠近陷阱的动作价值是否变低。
5. 多次随机种子下是否大致都能收敛。

RL 的随机性比监督学习更强。一次训练成功不代表算法完全可靠；多跑几次、观察趋势，比只看单次输出更有价值。

## 学习路线建议

建议按这个顺序掌握本节：先用图画出状态、动作和奖励；再手算一两个状态的贝尔曼更新；然后运行价值迭代，观察价值如何从目标向外传播；最后运行 Q-Learning，观察智能体如何从随机试错中学出类似策略。

如果你能解释“价值迭代需要知道环境规则，而 Q-Learning 只需要交互样本”，就已经抓住了 model-based 与 model-free 的核心差异。

## 算法对照：价值迭代 vs Q-Learning

价值迭代和 Q-Learning 都依赖贝尔曼思想，但使用条件不同。

| 方法 | 是否需要知道环境模型 | 学到什么 | 典型用途 |
|------|----------------------|----------|----------|
| 价值迭代 | 需要知道转移和奖励规则 | 最优状态价值与策略 | 小型、规则明确的问题 |
| Q-Learning | 不需要，只要能交互 | Q 表和贪心策略 | 无模型试错学习 |

如果你能提前枚举每个状态执行每个动作会到哪里，价值迭代很直接。如果环境像游戏或机器人一样只能通过尝试获得反馈，Q-Learning 更自然。

## 调参检查表

- $\alpha$ 太大：Q 值可能震荡；太小：学习很慢。
- $\gamma$ 太大：更重视长期奖励，但可能让学习更慢；太小：行为短视。
- $\epsilon$ 衰减太快：探索不足；衰减太慢：训练后期仍然随机。
- 每步惩罚太小：智能体可能绕路；太大：智能体可能过度保守。
- episode 最大步数太短：还没到目标就被截断；太长：无效探索浪费训练时间。

强化学习调参时不要一次改很多东西。每次只改一个因素，并记录平均奖励、成功率和最终路径。